# Task 13: Deploying Deep Learning Applications on Kubernetes
Run on a machine with Docker, kubectl and Kind (see README.md). Run cells top to bottom; the saved outputs are your results.

## 1. Build the container image (model is trained during the build)

In [ ]:
!docker --version && kubectl version --client && kind version
!docker build --provenance=false --sbom=false -t dl-digits:1.0 app 2>&1 | tail -n 15
!docker image ls dl-digits

## 2. Create the cluster (1 control-plane + 2 workers, NodePort 30090 -> localhost:8090)

In [ ]:
!kind create cluster --name task13 --config kind-config-task13.yaml --wait 120s
!kubectl config use-context kind-task13
!kubectl wait --for=condition=Ready nodes --all --timeout=180s
!kubectl get nodes -o wide

## 3. Load the image into the cluster

In [ ]:
!kind load docker-image dl-digits:1.0 --name task13   # if this fails: docker save dl-digits:1.0 -o /tmp/dl.tar && kind load image-archive /tmp/dl.tar --name task13

## 4. Deployment manifests and services

In [ ]:
!cat k8s13/*.yaml

## 5. Deploy and expose

In [ ]:
!kubectl apply -f k8s13/
!kubectl rollout status deployment/dl-api -n dl-demo --timeout=240s

## 6. Verify objects

In [ ]:
!kubectl get pods -n dl-demo -o wide
!kubectl get deploy,svc,endpointslices,cm -n dl-demo
!kubectl describe svc dl-api -n dl-demo

## 7. Verify accessibility from outside the cluster

In [ ]:
import json, urllib.request
get = lambda p: json.loads(urllib.request.urlopen('http://localhost:8090'+p, timeout=10).read())
print(json.dumps(get('/health'), indent=2))
print(json.dumps(get('/model-info'), indent=2))
for i in range(10):
    r = get(f'/predict/sample/{i}')
    print(i, r['prediction'], r['true_label'], r['correct'], r['pod'])

In [ ]:
from collections import Counter
print(Counter(get('/health')['pod'] for _ in range(30)))   # load balancing across pods

## 8. Verify accessibility from inside the cluster (ClusterIP)

In [ ]:
!kubectl run curl-test -n dl-demo --rm -i --restart=Never --image=curlimages/curl:8.10.1 -- curl -s http://dl-api-internal/health

## 9. Monitor

In [ ]:
!kubectl logs deployment/dl-api -n dl-demo --tail=12
!kubectl describe deployment dl-api -n dl-demo
!kubectl get events -n dl-demo --sort-by=.lastTimestamp

## 10. Cleanup (optional)

In [ ]:
# !kind delete cluster --name task13

## Observations
*(Write 4-6 lines in your own words about what you saw.)*